# Model Evaluation — Austrian Tax Law Q&A

Evaluate 3 models on 643 Austrian tax law questions using multiple metrics:
- **Reference-free**: answer length, repetition rate, vocabulary diversity
- **Cross-model** (Model 3 as pseudo-reference): BLEU, ROUGE-1, ROUGE-2, ROUGE-L
- **Semantic similarity**: BERTScore
- **Exact Match**
- **Error analysis**: qualitative inspection of failure modes

In [ ]:
!pip install rouge-score nltk bert-score pandas numpy matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load model results
m1 = pd.read_csv('../results/model1_results.csv')
m2 = pd.read_csv('../results/model2_results.csv')
m3 = pd.read_csv('../results/model3_results.csv')

# Load questions
questions = pd.read_csv('../dataset_clean.csv')

# Fill NaN answers with empty string
for df in [m1, m2, m3]:
    df['answer'] = df['answer'].fillna('')

print(f'Model 1 (Baseline GPT-2): {len(m1)} answers')
print(f'Model 2 (Fine-tuned GPT-2): {len(m2)} answers')
print(f'Model 3 (RAG + GPT-4o-mini): {len(m3)} answers')
print(f'Questions: {len(questions)}')

## 1. Reference-Free Metrics

Basic quality indicators that don't require ground truth answers.

In [ ]:
def compute_reference_free_metrics(answers):
    """Compute reference-free quality metrics for a list of answers."""
    lengths = [len(a) for a in answers]
    word_counts = [len(a.split()) for a in answers]
    
    # Repetition rate: ratio of unique trigrams to total trigrams (higher = less repetitive)
    rep_rates = []
    for a in answers:
        words = a.split()
        if len(words) < 3:
            rep_rates.append(0.0)
            continue
        trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
        rep_rates.append(len(set(trigrams)) / len(trigrams) if trigrams else 0.0)
    
    # Vocabulary diversity: unique words / total words
    all_words = ' '.join(answers).split()
    vocab_diversity = len(set(all_words)) / len(all_words) if all_words else 0.0
    
    # Empty/very short answers
    empty_count = sum(1 for a in answers if len(a.strip()) <= 5)
    
    return {
        'avg_char_length': np.mean(lengths),
        'avg_word_count': np.mean(word_counts),
        'median_word_count': np.median(word_counts),
        'avg_trigram_uniqueness': np.mean(rep_rates),
        'vocabulary_diversity': vocab_diversity,
        'empty_or_trivial (<=5 chars)': empty_count,
        'empty_pct': f'{empty_count/len(answers)*100:.1f}%'
    }

models = {'Model 1 (Baseline)': m1, 'Model 2 (Fine-tuned)': m2, 'Model 3 (RAG)': m3}
ref_free = {name: compute_reference_free_metrics(df['answer'].tolist()) for name, df in models.items()}

ref_free_df = pd.DataFrame(ref_free).T
ref_free_df

In [ ]:
# Visualize answer length distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, df) in enumerate(models.items()):
    word_counts = df['answer'].apply(lambda x: len(x.split()))
    axes[i].hist(word_counts, bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(name)
    axes[i].set_xlabel('Word count')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(word_counts.median(), color='red', linestyle='--', label=f'Median: {word_counts.median():.0f}')
    axes[i].legend()
plt.suptitle('Answer Length Distribution (Words)', fontsize=14)
plt.tight_layout()
plt.savefig('../evaluation/answer_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. BLEU Score

BLEU (Bilingual Evaluation Understudy) measures n-gram precision between generated text and a reference.
Since no ground truth exists, we use **Model 3 (RAG)** as pseudo-reference — it uses actual law text and GPT-4o-mini,
making it the most factually grounded model.

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def compute_bleu(predictions, references):
    """Compute average sentence-level BLEU score."""
    smooth = SmoothingFunction().method1
    scores = []
    for pred, ref in zip(predictions, references):
        ref_tokens = ref.split()
        pred_tokens = pred.split()
        if len(ref_tokens) == 0 or len(pred_tokens) == 0:
            scores.append(0.0)
            continue
        score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
        scores.append(score)
    return np.mean(scores), scores

ref_answers = m3['answer'].tolist()

bleu1_avg, bleu1_scores = compute_bleu(m1['answer'].tolist(), ref_answers)
bleu2_avg, bleu2_scores = compute_bleu(m2['answer'].tolist(), ref_answers)
bleu3_avg, _ = compute_bleu(ref_answers, ref_answers)  # self-reference = upper bound

print(f'BLEU Scores (vs Model 3 as reference):')
print(f'  Model 1 (Baseline):   {bleu1_avg:.4f}')
print(f'  Model 2 (Fine-tuned): {bleu2_avg:.4f}')
print(f'  Model 3 (Self-ref):   {bleu3_avg:.4f}')

## 3. ROUGE Scores

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) measures recall of n-grams.
- **ROUGE-1**: unigram overlap
- **ROUGE-2**: bigram overlap
- **ROUGE-L**: longest common subsequence

In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(predictions, references):
    """Compute average ROUGE-1, ROUGE-2, ROUGE-L F1 scores."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred.strip() or not ref.strip():
            r1.append(0.0); r2.append(0.0); rl.append(0.0)
            continue
        scores = scorer.score(ref, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)
    return {'ROUGE-1': np.mean(r1), 'ROUGE-2': np.mean(r2), 'ROUGE-L': np.mean(rl)}

rouge_m1 = compute_rouge(m1['answer'].tolist(), ref_answers)
rouge_m2 = compute_rouge(m2['answer'].tolist(), ref_answers)
rouge_m3 = compute_rouge(ref_answers, ref_answers)

rouge_df = pd.DataFrame({'Model 1 (Baseline)': rouge_m1, 'Model 2 (Fine-tuned)': rouge_m2, 'Model 3 (Self-ref)': rouge_m3}).T
rouge_df

## 4. BERTScore

BERTScore uses contextual embeddings from a pre-trained BERT model to compute semantic similarity
between predictions and references. Unlike BLEU/ROUGE, it captures meaning beyond surface-level n-gram overlap.

In [ ]:
from bert_score import score as bert_score

# Use multilingual BERT since answers are in German
print('Computing BERTScore (this may take a few minutes)...')

P1, R1, F1_m1 = bert_score(m1['answer'].tolist(), ref_answers, lang='de', verbose=False)
P2, R2, F1_m2 = bert_score(m2['answer'].tolist(), ref_answers, lang='de', verbose=False)

print(f'BERTScore F1 (vs Model 3 as reference):')
print(f'  Model 1 (Baseline):   {F1_m1.mean():.4f}')
print(f'  Model 2 (Fine-tuned): {F1_m2.mean():.4f}')
print(f'  Model 3 (Self-ref):   1.0000')

## 5. Exact Match

Exact Match checks if the prediction is identical to the reference (after normalization).
This is a very strict metric — even semantically correct answers score 0 if worded differently.

In [ ]:
def normalize(text):
    """Normalize text for exact match comparison."""
    return ' '.join(text.lower().split())

def exact_match_rate(predictions, references):
    matches = sum(1 for p, r in zip(predictions, references) if normalize(p) == normalize(r))
    return matches / len(predictions)

em_m1 = exact_match_rate(m1['answer'].tolist(), ref_answers)
em_m2 = exact_match_rate(m2['answer'].tolist(), ref_answers)
em_m3 = exact_match_rate(ref_answers, ref_answers)

print(f'Exact Match Rate (vs Model 3 as reference):')
print(f'  Model 1 (Baseline):   {em_m1:.4f}')
print(f'  Model 2 (Fine-tuned): {em_m2:.4f}')
print(f'  Model 3 (Self-ref):   {em_m3:.4f}')

## 6. Main Results Table

In [ ]:
# Compile all metrics into a single results table
results = pd.DataFrame({
    'Model': ['Model 1 (Baseline GPT-2)', 'Model 2 (Fine-tuned GPT-2)', 'Model 3 (RAG + GPT-4o-mini)'],
    'Avg Words': [
        ref_free_df.loc['Model 1 (Baseline)', 'avg_word_count'],
        ref_free_df.loc['Model 2 (Fine-tuned)', 'avg_word_count'],
        ref_free_df.loc['Model 3 (RAG)', 'avg_word_count']
    ],
    'Trigram Uniqueness': [
        ref_free_df.loc['Model 1 (Baseline)', 'avg_trigram_uniqueness'],
        ref_free_df.loc['Model 2 (Fine-tuned)', 'avg_trigram_uniqueness'],
        ref_free_df.loc['Model 3 (RAG)', 'avg_trigram_uniqueness']
    ],
    'BLEU': [bleu1_avg, bleu2_avg, bleu3_avg],
    'ROUGE-1': [rouge_m1['ROUGE-1'], rouge_m2['ROUGE-1'], rouge_m3['ROUGE-1']],
    'ROUGE-2': [rouge_m1['ROUGE-2'], rouge_m2['ROUGE-2'], rouge_m3['ROUGE-2']],
    'ROUGE-L': [rouge_m1['ROUGE-L'], rouge_m2['ROUGE-L'], rouge_m3['ROUGE-L']],
    'BERTScore F1': [F1_m1.mean().item(), F1_m2.mean().item(), 1.0],
    'Exact Match': [em_m1, em_m2, em_m3]
})

# Format numeric columns
for col in results.columns[1:]:
    results[col] = results[col].apply(lambda x: f'{x:.4f}')

results.set_index('Model')

In [ ]:
# Save results table as CSV for the report
results.to_csv('../evaluation/evaluation_results.csv', index=False)
print('Saved evaluation_results.csv')

## 7. Metric Comparison Visualization

In [ ]:
# Bar chart comparing key metrics across models
metrics_to_plot = ['BLEU', 'ROUGE-1', 'ROUGE-L', 'BERTScore F1']
model_names = ['Model 1\n(Baseline)', 'Model 2\n(Fine-tuned)', 'Model 3\n(RAG)']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_to_plot))
width = 0.25

values = results.set_index('Model')
for i, (model, label) in enumerate(zip(values.index, model_names)):
    vals = [float(values.loc[model, m]) for m in metrics_to_plot]
    ax.bar(x + i*width, vals, width, label=label)

ax.set_ylabel('Score')
ax.set_title('Model Comparison — Key Metrics (vs Model 3 as Reference)')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../evaluation/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Error Analysis

Qualitative inspection of failure modes across models.

In [ ]:
# Sample 5 questions and compare answers across models
sample_ids = ['CORP-TAX-001', 'TAX-INTL-015', 'VAT-DOM-015', 'ESTG-NEW-010', 'SELF-014']

for qid in sample_ids:
    q_row = questions[questions['id'] == qid]
    if q_row.empty:
        continue
    question = q_row.iloc[0]['prompt']
    a1 = m1[m1['id'] == qid].iloc[0]['answer'] if qid in m1['id'].values else 'N/A'
    a2 = m2[m2['id'] == qid].iloc[0]['answer'] if qid in m2['id'].values else 'N/A'
    a3 = m3[m3['id'] == qid].iloc[0]['answer'] if qid in m3['id'].values else 'N/A'
    
    print(f'=== {qid} ===')
    print(f'Q: {question[:120]}...')
    print(f'Model 1: {a1[:150]}...')
    print(f'Model 2: {a2[:150]}...')
    print(f'Model 3: {a3[:150]}...')
    print()

In [ ]:
# Classify error types for Model 1 and Model 2
def classify_errors(answers):
    """Classify common error types in model answers."""
    errors = {'trivial_short': 0, 'repetitive': 0, 'off_topic': 0, 'reasonable': 0}
    
    for a in answers:
        words = a.split()
        # Trivial/empty answer (<=5 words)
        if len(words) <= 5:
            errors['trivial_short'] += 1
            continue
        # Repetitive: trigram uniqueness < 0.3
        trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
        uniqueness = len(set(trigrams)) / len(trigrams) if trigrams else 1.0
        if uniqueness < 0.3:
            errors['repetitive'] += 1
            continue
        errors['reasonable'] += 1
    
    return errors

error_types = {}
for name, df in models.items():
    error_types[name] = classify_errors(df['answer'].tolist())

error_df = pd.DataFrame(error_types).T
error_df['total'] = error_df.sum(axis=1)
print('Error classification (count):')
print(error_df)
print()

# Percentages
error_pct = error_df.div(error_df['total'], axis=0) * 100
error_pct = error_pct.drop(columns='total').round(1)
print('Error classification (%):')
print(error_pct)

In [ ]:
# Error type visualization
fig, ax = plt.subplots(figsize=(10, 5))
error_pct.plot(kind='bar', ax=ax, rot=0)
ax.set_ylabel('Percentage of answers (%)')
ax.set_title('Error Type Distribution by Model')
ax.legend(title='Error Type')
plt.tight_layout()
plt.savefig('../evaluation/error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Evaluation complete. Files saved to evaluation/ folder.')